# Getting started with DragonScope

DragonScope lets you see, edit and audit the memory of a BDH (Dragon Hatchling) model. This notebook takes about a minute on a laptop CPU.

It uses the small demo model in `models/`. To use your own, pass any `BDH` model from [pathwaycom/bdh](https://github.com/pathwaycom/bdh).

In [1]:
import dragonscope as ds

model = ds.load_model("../models/best_model.pt")

## 1. Capture memory

`capture` wraps the model. Everything it reads or writes enters its memory, and each byte is recorded.

In [2]:
memory = ds.capture(model)
memory.read("U: My card number is 4242 4242 4242 4242.\nA: Noted.\n")
memory.read("U: My branch is Kandy.\nA: Noted.\n")

step = memory.history[-1]
print(f"{len(memory.history)} bytes recorded")
print(f"Neurons firing in each round of the last byte: {step.firing}")
print(f"Neurons passing a signal on: {step.passing}")
print(f"Memory shape (round, head, dimension, neuron): {tuple(memory.rho.shape)}")

85 bytes recorded
Neurons firing in each round of the last byte: [2071, 1013, 941, 956]
Neurons passing a signal on: [532, 151, 84, 118]
Memory shape (round, head, dimension, neuron): (4, 4, 128, 1024)


The memory `rho` is a compressed form of the model's synaptic state. Pass `keep_states=True` to `capture` to keep a copy at every byte, though that uses about 8 MB a byte for this model.

The model's weights never change here. Only this memory does.

## 2. Ask it

`fork` makes an independent copy, so you can try things without changing the session.

In [3]:
trial = memory.fork()
trial.read("U: What is my card number?\nA: ")
print(trial.generate())

Your card number is 4242 4242 4242 4242.



## 3. Forget

`forget` removes every copy of a value. `subtract` removes the bytes' own traces, and `rebuild` rereads the session without them.

In [4]:
edited = memory.fork()
result = edited.forget("4242 4242 4242 4242", method="subtract")
print(f"Erased {result.bytes_erased} bytes in {result.seconds * 1000:.1f} ms")
print(f"Distance from never having read them: {result.residual:.2%}")
print("By round:", [f"{value:.2%}" for value in result.residual_per_round])

edited.read("U: What is my card number?\nA: ")
print(edited.generate())

Erased 19 bytes in 30.6 ms
Distance from never having read them: 6.49%
By round: ['0.00%', '6.97%', '10.27%', '8.98%']


I don't have that on record.



`subtract` leaves a residual, starting from the second round, because later bytes absorbed some of what was erased. `rebuild` leaves none:

In [5]:
rebuilt = memory.fork()
result = rebuilt.forget("4242 4242 4242 4242", method="rebuild")
print(f"{result.seconds * 1000:.1f} ms, distance {result.residual:.2%}")

42.0 ms, distance 0.00%


## 4. Measure whether it worked

A forgetting report tests your own probes before forgetting, after `subtract` and after `rebuild`. It also checks that the probes could recall the value in the first place, because a clean result would mean nothing otherwise.

In [6]:
from IPython.display import Markdown

case = ds.Case(
    text="U: My card number is 5555 5555 5555 4444.\nA: Noted.\nU: My branch is Kandy.\nA: Noted.\n",
    forget=["5555 5555 5555 4444"],
    probes=[
        ds.Probe("U: What is my card number?\nA: ", expect="5555 5555 5555 4444", about="forgotten"),
        ds.Probe("U: What is my branch?\nA: ", expect="Kandy", about="kept"),
    ],
)
report = ds.forgetting_report(model, [case])
Markdown(report.to_markdown())

# Forgetting report: PASS

1 conversations and 2 probes.

| Measure | Before | `subtract` | `rebuild` (reference) |
|---|---|---|---|
| Recall of forgotten values | 100.0% | 0.0% | 0.0% |
| Recall of kept values | 100.0% | 100.0% | 100.0% |
| Time to forget | n/a | 38.3 ms | 77.1 ms |
| Distance from never reading it | n/a | 6.6% | 0.0% |

Replies from `subtract` matching `rebuild`: 100.0%.
Next-byte distance from `rebuild`: 0.0%.

| Check | Result |
|---|---|
| Probes could recall the value before forgetting | pass |
| `subtract`: forgotten value no longer recalled | pass |
| `rebuild`: forgotten value no longer recalled | pass |
| `subtract`: other values kept | pass |
| `rebuild`: other values kept | pass |

This covers these probes on this model. It is evidence, not a proof.

Use many cases for a meaningful result. A report covers the probes it was given on this model, so it is evidence and not a proof.

## 5. Open the dashboard

This shows the session live: neurons firing, watched text, forgetting and reports. It needs the dashboard extra: `pip install -e ".[dashboard]"` from the repository, and uses address http://127.0.0.1:8100.

```python
ds.dashboard(memory)
```

Or, from a terminal:

```bash
dragonscope-dashboard --model models/best_model.pt
```